# AE Triage Scenario C: Dataset + Experiments + V2 Traces

Implements "Same agent after a change": create a dataset of triggered-issue cases, create an experiment with v1 and v2 runs, and send new OTLP traces for v2 so both the experiment comparison and the trace UI show the version change.

**Requirements:** `arize>=8` (for ArizeClient, datasets, experiments) and `arize-otel` (for tracing). If you have arize 7.x, run: `pip install "arize>=8" arize-otel`

In [1]:
# Uncomment if you need arize v8 (ArizeClient, datasets, experiments) and arize-otel:
#!pip install -q "arize>=8" arize-otel

In [2]:
import os
# Credentials (same as synthetic_spans_ae_triage.ipynb)
space_id = os.environ["ARIZE_SPACE_ID"]
api_key = os.environ["ARIZE_API_KEY"]
project_name = "ae_triage_agent"

# Set this after the first run to reuse the same dataset (avoids 409 "name already exists")
DATASET_ID = os.environ.get("ARIZE_DATASET_ID")  # Set to None to create a new dataset

In [3]:
import pandas as pd
import json

# Same synthetic data as AE triage notebook (one report / expected classification for all cases)
AE_REPORT_RAW = (
    "62-year-old female receiving darzumab for DLBCL reported Grade 3 cytokine release syndrome "
    "on Day 2 of Cycle 1. Patient hospitalized, treated with tocilizumab. Symptoms resolved within 48 hours."
)

EXPECTED_CLASSIFICATION = {
    "seriousness": "serious",
    "seriousness_criteria": ["hospitalization"],
    "expectedness": "expected",
    "causality": "possibly_related",
    "meddra_pt": "Cytokine release syndrome",
    "meddra_soc": "Immune system disorders",
}

# 20 case IDs: Scenario A (04821–04830), Scenario B (04831–04840)
CASE_IDS_A = [f"AE-2025-048{20 + i}" for i in range(1, 11)]
CASE_IDS_B = [f"AE-2025-048{30 + i}" for i in range(1, 11)]
ALL_CASE_IDS = CASE_IDS_A + CASE_IDS_B

## Step 1: Create or reuse the dataset

If **DATASET_ID** is set in the credentials cell, that dataset is used and creation is skipped. Otherwise a new dataset is created; copy the printed id into **DATASET_ID** for future runs.

In [4]:
from arize import ArizeClient

client = ArizeClient(api_key=api_key)

if DATASET_ID:
    dataset_id = DATASET_ID
    print(f"Using existing dataset: {dataset_id}")
else:
    # Do not include "id" — server assigns example ids; we get them from list_examples below.
    dataset_df = pd.DataFrame([
        {
            "report_text": AE_REPORT_RAW,
            "expected_classification": json.dumps(EXPECTED_CLASSIFICATION),
        }
        for case_id in ALL_CASE_IDS
    ])
    dataset = client.datasets.create(
        name="AE Triage Triggered Issues",
        space_id=space_id,
        examples=dataset_df,
    )
    dataset_id = dataset.id
    print(f"Created dataset: {dataset_id}")
    print(f"To reuse next time, set DATASET_ID = {repr(dataset_id)} in the credentials cell above.")

Using existing dataset: <redacted>


In [5]:
# Get example IDs (server may use our 'id' column or assign its own; list_examples returns the canonical ids)
examples_resp = client.datasets.list_examples(dataset_id=dataset_id, all=True)
examples_df = examples_resp.to_df()
example_ids = examples_df["id"].tolist()
print(f"Example IDs count: {len(example_ids)}")

  arize.pre_releases | WARNING | [BETA] datasets.list_examples is a beta API in Arize SDK v8.4.0 and may change without notice.
  arize.pre_releases | WARNING | [BETA] datasets.get is a beta API in Arize SDK v8.4.0 and may change without notice.


Example IDs count: 20


## Step 2: Create two experiments (with issues vs improved)

Two separate experiments, each with 20 runs (one per dataset example). Use **evaluator_columns** so each run includes an eval (score/label/explanation) representing issues vs no exceptions. Compare them in the Arize dashboard.

In [6]:
from arize.experiments.types import ExperimentTaskFieldNames
from arize.experiments import EvaluationResultFieldNames

task_fields = ExperimentTaskFieldNames(example_id="example_id", output="output")
evaluator_columns = {
    "triage_outcome": EvaluationResultFieldNames(
        score="eval_score",
        label="eval_label",
        explanation="eval_explanation",
    )
}

# Experiment 1: v1 with exceptions/issues (policy blocks, recovery failures)
output_v1 = json.dumps({**EXPECTED_CLASSIFICATION, "had_block": True, "disposition": "escalate"})
runs_v1 = []
for ex_id in example_ids:
    runs_v1.append({
        "example_id": ex_id,
        "output": output_v1,
        "prompt_version": "v1",
        "model": "gpt-4o",
        "eval_score": 0.0,
        "eval_label": "exception_or_block",
        "eval_explanation": "Policy blocked; recovery failed or missing disposition.",
    })

experiment_v1 = client.experiments.create(
    name="AE Triage v1 (with issues)",
    dataset_id=dataset_id,
    experiment_runs=runs_v1,
    task_fields=task_fields,
    evaluator_columns=evaluator_columns,
)
print(f"Created experiment (v1 with issues): {experiment_v1.id}")

# Experiment 2: v2 improved — no exceptions, clean classification
output_v2 = json.dumps(EXPECTED_CLASSIFICATION)
runs_v2 = []
for ex_id in example_ids:
    runs_v2.append({
        "example_id": ex_id,
        "output": output_v2,
        "prompt_version": "v2",
        "model": "gpt-4o",
        "eval_score": 1.0,
        "eval_label": "pass",
        "eval_explanation": "No exceptions; clean classification.",
    })

experiment_v2 = client.experiments.create(
    name="AE Triage v2 (improved)",
    dataset_id=dataset_id,
    experiment_runs=runs_v2,
    task_fields=task_fields,
    evaluator_columns=evaluator_columns,
)
print(f"Created experiment (v2 improved): {experiment_v2.id}")

  arize.pre_releases | WARNING | [BETA] experiments.create is a beta API in Arize SDK v8.4.0 and may change without notice.
Created experiment (v1 with issues): RXhwZXJpbWVudDo2MDE4MTo4Qlo0
Created experiment (v2 improved): RXhwZXJpbWVudDo2MDE4MjpWa0x0


## Step 3: Send new traces for v2 (Trace UI)

Datasets/experiments and traces are separate in Arize. Emit OTLP traces with `llm.prompt_template.version="v2"` so the "after change" behavior is visible in the Trace UI.

In [7]:
from arize.otel import register
from opentelemetry import trace
from opentelemetry.trace.status import Status, StatusCode
import time
import random

# Same PARSED_INTAKE, MEDDRA_TOP_3, SOP_TOP_3 as main notebook (abbreviated here)
PARSED_INTAKE = {
    "patient_age": 62,
    "patient_sex": "female",
    "drug": "darzumab",
    "indication": "DLBCL",
    "event_description": "Grade 3 cytokine release syndrome",
    "onset": "Day 2 of Cycle 1",
    "outcome": "Symptoms resolved within 48 hours",
    "hospitalization": True,
    "treatment": "tocilizumab",
}
MEDDRA_TOP_3 = [
    {"document_id": "meddra_10001234", "content": "Cytokine release syndrome...", "score": 0.94},
    {"document_id": "meddra_10023456", "content": "Infusion related reaction...", "score": 0.78},
    {"document_id": "meddra_10034567", "content": "ICANS...", "score": 0.71},
]
SOP_TOP_3 = [
    {"document_id": "SOP-PV-003-v2.1-§4.2", "content": "Seriousness classification...", "score": 0.91},
    {"document_id": "SOP-PV-004-v1.2-§3.1", "content": "Expectedness...", "score": 0.87},
    {"document_id": "SOP-PV-005-v1.0-§2", "content": "Auto-submission policy...", "score": 0.82},
]

tracer_provider = register(
    space_id=space_id,
    api_key=api_key,
    project_name=project_name,
)
tracer = trace.get_tracer(__name__)

🔭 OpenTelemetry Tracing Details 🔭
|  Arize Project: ae_triage_agent
|  Span Processor: BatchSpanProcessor
|  Collector Endpoint: otlp.arize.com
|  Transport: gRPC
|  Transport Headers: {'authorization': '****', 'api_key': '****', 'arize-space-id': '****', 'space_id': '****', 'arize-interface': '****'}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [8]:
def build_trace_scenario_c_v2(case_id: str, session_id: str):
    """Scenario C v2: same shape as Scenario A but llm.prompt_template.version='v2'. Success path only."""
    chain_name = "AE Triage Agent"
    latency_ms = random.uniform(50, 150)
    time.sleep(0.002)

    with tracer.start_as_current_span(chain_name) as chain_span:
        chain_span.set_attribute("openinference.span.kind", "CHAIN")
        chain_span.set_attribute("input.value", AE_REPORT_RAW)
        chain_span.set_attribute("input.mime_type", "text/plain")
        chain_span.set_attribute("session.id", session_id)
        chain_span.set_attribute("user.id", "pv_analyst_001")
        chain_span.set_attribute("metadata.case_id", case_id)

        with tracer.start_as_current_span("Report Intake Agent") as agent1_span:
            agent1_span.set_attribute("openinference.span.kind", "AGENT")
            agent1_span.set_attribute("agent.name", "report_intake_agent")
            agent1_span.set_attribute("input.value", AE_REPORT_RAW)
            agent1_span.set_attribute("input.mime_type", "text/plain")

            with tracer.start_as_current_span("AE Report Intake") as span1:
                span1.set_attribute("openinference.span.kind", "CHAIN")
                span1.set_attribute("input.value", AE_REPORT_RAW)
                span1.set_attribute("output.value", json.dumps(PARSED_INTAKE))
                span1.set_attribute("output.mime_type", "application/json")
                span1.set_status(Status(StatusCode.OK))

                with tracer.start_as_current_span("MedDRA Term Lookup") as span2:
                    span2.set_attribute("openinference.span.kind", "RETRIEVER")
                    span2.set_attribute("input.value", "cytokine release syndrome grade 3")
                    span2.set_attribute("output.value", json.dumps([{"document_id": d["document_id"], "score": d["score"]} for d in MEDDRA_TOP_3]))
                    span2.set_attribute("output.mime_type", "application/json")
                    for idx, doc in enumerate(MEDDRA_TOP_3):
                        span2.set_attribute(f"retrieval.documents.{idx}.document.id", doc["document_id"])
                        span2.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                        span2.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
                    span2.set_status(Status(StatusCode.OK))

                with tracer.start_as_current_span("SOP & Product Label Retrieval") as span3:
                    span3.set_attribute("openinference.span.kind", "RETRIEVER")
                    span3.set_attribute("input.value", "darzumab cytokine release syndrome seriousness classification")
                    span3.set_attribute("output.value", json.dumps([{"document_id": d["document_id"], "score": d["score"]} for d in SOP_TOP_3]))
                    span3.set_attribute("output.mime_type", "application/json")
                    for idx, doc in enumerate(SOP_TOP_3):
                        span3.set_attribute(f"retrieval.documents.{idx}.document.id", doc["document_id"])
                        span3.set_attribute(f"retrieval.documents.{idx}.document.content", doc["content"])
                        span3.set_attribute(f"retrieval.documents.{idx}.document.score", doc["score"])
                    span3.set_status(Status(StatusCode.OK))

            agent1_span.set_attribute("output.value", json.dumps(PARSED_INTAKE))
            agent1_span.set_attribute("output.mime_type", "application/json")

        with tracer.start_as_current_span("Triage Classification Agent") as agent2_span:
            agent2_span.set_attribute("openinference.span.kind", "AGENT")
            agent2_span.set_attribute("agent.name", "triage_classification_agent")
            agent2_span.set_attribute("input.value", AE_REPORT_RAW[:500])
            agent2_span.set_attribute("input.mime_type", "text/plain")

            # Classification Reasoning (LLM) — v2 prompt version, success only
            with tracer.start_as_current_span("Classification Reasoning") as span4:
                span4.set_attribute("openinference.span.kind", "LLM")
                span4.set_attribute("llm.model_name", "gpt-4o")
                span4.set_attribute("llm.provider", "openai")
                span4.set_attribute("llm.invocation_parameters", json.dumps({"temperature": 0.0}))
                span4.set_attribute("llm.input_messages.0.message.role", "system")
                span4.set_attribute("llm.input_messages.0.message.content", "You are a PV triage assistant (v2). Classify the AE using MedDRA and SOP context.")
                span4.set_attribute("llm.input_messages.1.message.role", "user")
                span4.set_attribute("llm.input_messages.1.message.content", f"Report: {AE_REPORT_RAW[:500]}... [Retrieved MedDRA and SOP chunks omitted for brevity]")
                span4.set_attribute("llm.output_messages.0.message.role", "assistant")
                span4.set_attribute("llm.output_messages.0.message.content", json.dumps(EXPECTED_CLASSIFICATION))
                span4.set_attribute("llm.token_count.prompt", 1800)
                span4.set_attribute("llm.token_count.completion", 120)
                span4.set_attribute("llm.token_count.total", 1920)
                span4.set_attribute("input.value", AE_REPORT_RAW[:500])
                span4.set_attribute("output.value", json.dumps(EXPECTED_CLASSIFICATION))
                # v2: different template and version
                span4.set_attribute("llm.prompt_template.template", "You are a PV triage assistant (v2). Classify the AE using MedDRA and SOP context.\n\nReport: {report_text}")
                span4.set_attribute("llm.prompt_template.version", "v2")
                span4.set_attribute("llm.prompt_template.variables", json.dumps({"report_text": AE_REPORT_RAW[:500] + "..."}))
                span4.set_status(Status(StatusCode.OK))

            agent2_span.set_attribute("output.value", json.dumps(EXPECTED_CLASSIFICATION))
            agent2_span.set_attribute("output.mime_type", "application/json")

        classification_summary = json.dumps(EXPECTED_CLASSIFICATION)
        chain_span.set_attribute("output.value", classification_summary[:500] if len(classification_summary) > 500 else classification_summary)
        chain_span.set_attribute("output.mime_type", "application/json")
        chain_span.set_attribute("latency_ms", round(latency_ms, 2))
        chain_span.set_status(Status(StatusCode.OK))

In [9]:
NUM_V2_TRACES = 8
session_id_base = "session_ae_c_v2"

for i in range(1, NUM_V2_TRACES + 1):
    case_id = ALL_CASE_IDS[(i - 1) % len(ALL_CASE_IDS)]
    session_id = f"{session_id_base}_{i:03d}"
    build_trace_scenario_c_v2(case_id=case_id, session_id=session_id)

flush_timeout = 15000
flush_success = trace.get_tracer_provider().force_flush(timeout_millis=flush_timeout)
print(f"Flush completed: {flush_success}. Sent {NUM_V2_TRACES} v2 traces.")

Flush completed: True. Sent 8 v2 traces.


## Step 4: Using Arize UI

- **Experiments:** Open the two experiments in your space: **AE Triage v1 (with issues)** and **AE Triage v2 (improved)**. Compare them using the **triage_outcome** evaluator (score/label): v1 runs have `eval_label="exception_or_block"` and score 0; v2 runs have `eval_label="pass"` and score 1. Use the dashboard to show fewer exceptions in v2.
- **Traces:** In the same project (**ae_triage_agent**), open the Trace UI. Filter or inspect spans by **`llm.prompt_template.version`** to see v1 (from Scenario A/B runs) vs **v2** (from the traces sent above).